In [0]:
STORAGE_ACCOUNT = 'stccasemauricioes'
CONTAINER = 'data'
STORAGE_KEY = 'COLOCAR_CHAVE_AQUI'

spark.conf.set("fs.azure.account.key." + STORAGE_ACCOUNT + ".blob.core.windows.net", STORAGE_KEY)

BASE_GOLD   = "wasbs://" + CONTAINER + "@" + STORAGE_ACCOUNT + ".blob.core.windows.net/gold"

from pyspark.sql import functions as F

# Carrega Gold e cria temp views pra usar SQL
spark.read.format('delta').load(f'{BASE_GOLD}/fact_vendas').createOrReplaceTempView('fact_vendas')
spark.read.format('delta').load(f'{BASE_GOLD}/dim_produto').createOrReplaceTempView('dim_produto')
spark.read.format('delta').load(f'{BASE_GOLD}/dim_loja').createOrReplaceTempView('dim_loja')
spark.read.format('delta').load(f'{BASE_GOLD}/dim_data').createOrReplaceTempView('dim_data')

print('Views criadas: fact_vendas, dim_produto, dim_loja, dim_data')

Views criadas: fact_vendas, dim_produto, dim_loja, dim_data


In [0]:
# %sql
print('1) Faturamento por regiao:')
spark.sql("""
    SELECT 
        l.loja_regiao,
        COUNT(DISTINCT f.pedido_id) AS pedidos,
        ROUND(SUM(f.valor_bruto), 2) AS faturamento,
        ROUND(AVG(f.valor_bruto), 2) AS ticket_medio_item
    FROM fact_vendas f
    JOIN dim_loja l USING (loja_id)
    GROUP BY l.loja_regiao
    ORDER BY faturamento DESC
""").show()

1) Faturamento por regiao:
+------------+-------+-----------+-----------------+
| loja_regiao|pedidos|faturamento|ticket_medio_item|
+------------+-------+-----------+-----------------+
|     Sudeste|   2146|  162275.96|            30.87|
|    Nordeste|   1148|   92101.06|            32.30|
|         Sul|    739|   56848.30|            31.03|
|Centro-Oeste|    335|   27222.53|            31.91|
+------------+-------+-----------+-----------------+



In [0]:
print('2) Top 10 produtos por faturamento:')
spark.sql("""
    SELECT 
        p.produto_nome,
        p.produto_marca,
        p.faixa_preco,
        SUM(f.quantidade) AS unidades,
        ROUND(SUM(f.valor_bruto), 2) AS faturamento
    FROM fact_vendas f
    JOIN dim_produto p USING (produto_id)
    GROUP BY p.produto_nome, p.produto_marca, p.faixa_preco
    ORDER BY faturamento DESC
    LIMIT 10
""").show(truncate=False)

2) Top 10 produtos por faturamento:
+----------------------------+-------------+-------------+--------+-----------+
|produto_nome                |produto_marca|faixa_preco  |unidades|faturamento|
+----------------------------+-------------+-------------+--------+-----------+
|Colorado Indica 600ml       |Colorado     |Super Premium|2241    |44586.93   |
|Patagonia Amber Lager 740ml |Patagonia    |Super Premium|2232    |36832.46   |
|Colorado Appia 600ml        |Colorado     |Super Premium|1921    |36338.36   |
|Goose Island IPA 355ml      |Goose Island |Premium      |2038    |30413.72   |
|Eisenbahn Pilsen 600ml      |Eisenbahn    |Premium      |2184    |28208.05   |
|Leffe Blonde 330ml          |Leffe        |Premium      |2197    |26105.79   |
|Hoegaarden 330ml            |Hoegaarden   |Premium      |1943    |21138.41   |
|Original 600ml              |Antarctica   |Padrao       |2007    |19884.59   |
|Corona Extra Long Neck 330ml|Corona       |Padrao       |2073    |17614.47   |
|Bec

In [0]:
print('3) Sazonalidade - dia da semana:')
spark.sql("""
    SELECT
        d.nome_dia_semana AS dia,
        COUNT(DISTINCT f.pedido_id) AS pedidos,
        ROUND(SUM(f.valor_bruto), 2) AS faturamento
    FROM fact_vendas f
    JOIN dim_data d USING (data_id)
    GROUP BY d.dia_semana, d.nome_dia_semana
    ORDER BY d.dia_semana
""").show()

3) Sazonalidade - dia da semana:
+-------+-------+-----------+
|    dia|pedidos|faturamento|
+-------+-------+-----------+
|Domingo|    638|   46798.71|
|Segunda|    377|   29294.84|
|  Terca|    449|   34289.50|
| Quarta|    518|   42690.53|
| Quinta|    573|   46659.69|
|  Sexta|    901|   67250.65|
| Sabado|    912|   71463.93|
+-------+-------+-----------+



In [0]:
print('4) Mix de canal por estado:')
spark.sql("""
    WITH ped AS (
        SELECT l.loja_estado, f.canal_venda, COUNT(DISTINCT f.pedido_id) AS pedidos
        FROM fact_vendas f
        JOIN dim_loja l USING (loja_id)
        GROUP BY l.loja_estado, f.canal_venda
    ),
    tot AS (
        SELECT loja_estado, SUM(pedidos) AS total FROM ped GROUP BY loja_estado
    )
    SELECT p.loja_estado, p.canal_venda, p.pedidos,
           ROUND(100.0 * p.pedidos / t.total, 1) AS pct
    FROM ped p JOIN tot t USING (loja_estado)
    ORDER BY p.loja_estado, p.pedidos DESC
""").show(30)

4) Mix de canal por estado:
+-----------+-----------+-------+----+
|loja_estado|canal_venda|pedidos| pct|
+-----------+-----------+-------+----+
|         BA|    APP_IOS|    139|34.9|
|         BA|APP_ANDROID|    133|33.4|
|         BA|        WEB|    126|31.7|
|         CE|    APP_IOS|    135|35.5|
|         CE|APP_ANDROID|    124|32.6|
|         CE|        WEB|    121|31.8|
|         DF|APP_ANDROID|    123|36.7|
|         DF|    APP_IOS|    119|35.5|
|         DF|        WEB|     93|27.8|
|         MG|APP_ANDROID|    132|37.6|
|         MG|    APP_IOS|    118|33.6|
|         MG|        WEB|    101|28.8|
|         PE|APP_ANDROID|    124|33.5|
|         PE|        WEB|    124|33.5|
|         PE|    APP_IOS|    122|33.0|
|         PR|APP_ANDROID|    129|37.4|
|         PR|        WEB|    112|32.5|
|         PR|    APP_IOS|    104|30.1|
|         RJ|        WEB|    259|37.9|
|         RJ|APP_ANDROID|    218|31.9|
|         RJ|    APP_IOS|    206|30.2|
|         RS|APP_ANDROID|    144|36.